In [5]:
import sqlite3
import pandas as pd

# Download the real Telco Churn dataset directly from a public URL
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

# Connect to (or create) the database file
conn = sqlite3.connect('customer_data.db')

#Save the entire dataset into your SQL database as a table named 'Customers'
df.to_sql('Customers', conn, if_exists='replace', index=False)

# Verify the size of the real database
check_df = pd.read_sql('SELECT COUNT(*) as Total_Customers FROM Customers', conn)
print(f"Success! Your database now has {check_df['Total_Customers'][0]} rows of real customer data.")

# Close the connection
conn.close()

Success! Your database now has 7043 rows of real customer data.


In [6]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

#Pull the data back from your database
conn = sqlite3.connect('customer_data.db')
df = pd.read_sql('SELECT * FROM Customers', conn)
conn.close()

# Data Cleaning
# The model needs numbers, not text. We convert 'Yes/No' columns to 1/0
le = LabelEncoder()
df['Churn'] = le.fit_transform(df['Churn']) # 1 = Yes, 0 = No

# Let's use 3 numeric columns for our first simple prediction
features = ['SeniorCitizen', 'tenure', 'MonthlyCharges']
X = df[features]
y = df['Churn']

# 3. Train/Test Split (We hide 20% of data to see if the model can 'guess' correctly)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Train the Machine Learning Model (Random Forest)
model = RandomForestClassifier()
model.fit(X_train, y_train)

#Checking how good the accuracy of the model is on the test data
accuracy = model.score(X_test, y_test)
print(f"Model Accuracy: {accuracy:.2%}")

Model Accuracy: 75.59%


In [8]:
# Ask the model to predict the probability of leaving for everyone
# predict_proba returns two numbers: [Chance of Staying, Chance of Leaving]
# We use [:, 1] to only grab the "Chance of Leaving" percentage
df['Churn_Probability'] = model.predict_proba(df[['SeniorCitizen', 'tenure', 'MonthlyCharges']])[:, 1]

# Convert the probability to a clean percentage (e.g., 0.85 -> 85%)
df['Churn_Risk_Score'] = (df['Churn_Probability'] * 100).round(2)

# Export this final, enhanced dataset to a CSV file for Power BI
export_filename = 'PowerBI_Churn_Data.csv'
df.to_csv(export_filename, index=False)

print(f"Success! Your data is saved as '{export_filename}' and is ready for Power BI.")

Success! Your data is saved as 'PowerBI_Churn_Data.csv' and is ready for Power BI.
